In [2]:
%pip install -q sentence-transformers psycopg2-binary pgvector pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import psycopg2
import pandas as pd
from sentence_transformers import SentenceTransformer
from pgvector.psycopg2 import register_vector

D:\konst\PycharmProjects\mirea_real\Meal_planner\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
conn = psycopg2.connect(
    host="localhost",
    port=5432,
    database="Meal_planner",
    user="postgres",
    password="postgres"
)
register_vector(conn)

In [5]:
with conn.cursor() as cur:
    cur.execute("SELECT version();")
    print(cur.fetchone()[0])

PostgreSQL 18.2 on x86_64-windows, compiled by msvc-19.44.35222, 64-bit


In [6]:
meals = pd.read_sql_query(
    """
    SELECT
        v.id,
        v.name,
        v.meal_type,
        m.cooking_time,
        m.servings,
        v.calories,
        v.protein,
        v.fat,
        v.carbs,
        v.diet_type,
        v.allergens,
        v.tags,
        m.cost,
        m.goal,
        concat_ws(
            ' ',
            v.rag_text,
            'Цель: ' || m.goal,
            'Тип питания: ' || m.diet_type,
            'Время приготовления: ' || m.cooking_time || ' минут',
            'Цена: ' || m.cost || ' рублей'
        ) AS rag_text
    FROM meal_rag_data AS v
    JOIN meals AS m ON m.id = v.id
    ORDER BY v.meal_type, v.name
    """,
    conn,
)
meals.head()

C:\Users\valentin\AppData\Local\Temp\ipykernel_26776\3918764923.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  meals = pd.read_sql_query(


,id,name,meal_type,cooking_time,servings,calories,protein,fat,carbs,diet_type,allergens,tags,cost,goal,rag_text
0,f4860cf0-c919-48d7-b343-a06eabdf846b,Авокадо-тост с томатом,breakfast,15,1,176.0,7.0,7.1,23.4,vegan,[глютен],"[низкокалорийный, завтрак, домашний]",40.1,lose_weight,Авокадо-тост с томатом. Авокадо-тост с томатом...
1,72f01681-78b4-41d0-b61f-398fffe80ecb,Авокадо-тост с яйцом,breakfast,10,1,390.0,18.0,20.0,34.0,vegetarian,"[яйца, глютен]","[сытный, быстрый, завтрак]",108.8,maintain,Авокадо-тост с яйцом. Цельнозерновой хлеб с ав...
2,b49decb5-06bb-429a-babd-2d3170790687,Банановые овсяные оладьи,breakfast,15,1,285.0,12.6,8.4,38.8,vegetarian,[яйца],"[низкокалорийный, завтрак, домашний]",25.8,lose_weight,Банановые овсяные оладьи. Банановые овсяные ол...
3,b0c56edb-86b0-4c96-ae05-30d1f3a2cdc5,Гречка с яйцом,breakfast,15,1,259.0,14.3,7.1,37.8,vegetarian,[яйца],"[низкокалорийный, завтрак, домашний]",41.0,lose_weight,Гречка с яйцом. Гречка с яйцом. Состав: Гречка...
4,7c5cb749-8b6e-41f2-aa7c-a06325e706d4,Картофельная фриттата,breakfast,15,1,214.0,14.6,10.8,13.8,vegetarian,[яйца],"[низкокалорийный, завтрак, домашний]",45.6,lose_weight,Картофельная фриттата. Картофельная фриттата. ...


In [7]:
print(f"Блюд: {len(meals)}")

Блюд: 95


In [8]:
print(meals.loc[0, "rag_text"])

Авокадо-тост с томатом. Авокадо-тост с томатом. Состав: Хлеб цельнозерновой, Авокадо, Помидор.. Тип: breakfast. Калории: 176.00. Белки: 7.00 г. Жиры: 7.10 г. Углеводы: 23.40 г. Тип питания: vegan. Теги: низкокалорийный, завтрак, домашний. Ингредиенты: Авокадо 35.00 г, Помидор 50.00 г, Хлеб цельнозерновой 45.00 г Цель: lose_weight Тип питания: vegan Время приготовления: 15 минут Цена: 40.10 рублей


In [9]:
model = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1446.34it/s]


In [10]:
embedding = model.encode("блюдо для похудения с высоким содержанием белка")
print(len(embedding))

384


In [11]:
embeddings = model.encode(
    meals["rag_text"].tolist(),
    normalize_embeddings=True,
    show_progress_bar=True
)

Batches: 100%|██████████| 3/3 [00:03<00:00,  1.24s/it]


In [12]:
print(embeddings.shape)

(95, 384)


In [13]:
embeddings[0][:10]

array([-0.02427557, -0.0040604 ,  0.05049882,  0.04504099,  0.00798446,
       -0.01011306,  0.02260683, -0.01015752, -0.00614507, -0.05394844],
      dtype=float32)

In [14]:
with conn.cursor() as cur:
    for meal_id, embedding in zip(meals["id"], embeddings):
        cur.execute(
            """
            UPDATE meals
            SET embedding = %s
            WHERE id = %s
            """,
            (embedding, meal_id)
        )
conn.commit()

In [15]:
pd.read_sql(
    """
    SELECT
        COUNT(*) AS total,
        COUNT(embedding) AS with_embedding
    FROM meals
    """,
    conn
)

C:\Users\valentin\AppData\Local\Temp\ipykernel_26776\1738806380.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql(


,total,with_embedding
0,95,95


In [16]:
import re

GOAL_ALIASES = {
    "maintain": "maintain",
    "поддерживать форму": "maintain",
    "поддержать форму": "maintain",
    "сохранить форму": "maintain",
    "lose_weight": "lose_weight",
    "сбросить вес": "lose_weight",
    "похудеть": "lose_weight",
    "gain_weight": "gain_weight",
    "набрать массу": "gain_weight",
    "набрать вес": "gain_weight",
}
GOAL_CALORIES = {"maintain": 1800, "lose_weight": 1400, "gain_weight": 2200}

DIET_ALIASES = {
    "omnivore": "omnivore", "всеядный": "omnivore",
    "vegetarian": "vegetarian", "вегетарианство": "vegetarian", "вегетарианец": "vegetarian",
    "vegan": "vegan", "веганство": "vegan", "веган": "vegan",
    "pescatarian": "pescatarian", "пескетарианство": "pescatarian", "пескетарианец": "pescatarian",
}
DIET_COMPATIBILITY = {
    "omnivore": ["omnivore", "vegetarian", "vegan", "pescatarian"],
    "vegetarian": ["vegetarian", "vegan"], "vegan": ["vegan"],
    "pescatarian": ["pescatarian", "vegetarian", "vegan"],
}
EXCLUSION_ROOTS = {
    "мяс": ["мяс", "курин", "индейк", "говядин", "свин", "баран"],
    "рыб": ["рыб", "лосос", "тунец", "треск", "форел"],
    "молок": ["молок", "молоч", "творог", "йогурт", "сыр", "сливк"],
    "молоч": ["молок", "молоч", "творог", "йогурт", "сыр", "сливк"],
    "орех": ["орех", "миндал", "фундук", "кешью", "грецк"], "арахис": ["арахис"],
    "яйц": ["яйц"], "глютен": ["глютен", "хлеб", "макарон", "пшениц"],
    "пшениц": ["глютен", "хлеб", "макарон", "пшениц"], "соя": ["соя", "соев"],
    "лактоз": ["молок", "молоч", "творог", "йогурт", "сыр", "сливк"],
}

def normalize_goal(value):
    if value is None:
        return None
    value = str(value).strip().casefold()
    return GOAL_ALIASES.get(value, value)

def normalize_diet(value):
    if value is None:
        return "omnivore"
    value = str(value).strip().casefold()
    return DIET_ALIASES.get(value, value)

def allowed_diet_types(value):
    diet = normalize_diet(value)
    return DIET_COMPATIBILITY.get(diet, [diet])

def restriction_patterns(restrictions):
    """Turn free-text restrictions into SQL LIKE patterns for allergens and products."""
    if not restrictions:
        return []
    fragments = re.split(r"[,;\n]+|\s+и\s+", str(restrictions).casefold())
    roots = set()
    for fragment in fragments:
        fragment = re.sub(r"^\s*(без|исключить|исключи|не употреблять|не есть|аллергия на|аллергия)\s+", "", fragment).strip()
        if not fragment:
            continue
        matched = False
        for key, variants in EXCLUSION_ROOTS.items():
            if key in fragment:
                roots.update(variants)
                matched = True
        if not matched:
            roots.add(fragment)
    return [f"%{root}%" for root in sorted(roots)]

def load_profile(email):
    profile = pd.read_sql_query(
        """
        SELECT u.id AS user_id, u.name, u.email, p.goal, p.meals_per_day,
               p.diet_type, p.restrictions, p.budget, p.max_cooking_time, p.preferences
        FROM users AS u
        JOIN profiles AS p ON p.user_id = u.id
        WHERE u.email = %s
        """,
        conn,
        params=(email,),
    )
    if profile.empty:
        raise ValueError(f"Профиль для {email!r} не найден в базе Meal_planner")
    return profile.iloc[0].to_dict()

In [17]:
profile_email = "ivan@example.com"
profile = load_profile(profile_email)
pd.Series(profile)

C:\Users\valentin\AppData\Local\Temp\ipykernel_26776\1697811368.py:75: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  profile = pd.read_sql_query(


user_id             0d3ef8c9-4b8b-4fb8-ac63-629a0f3f9c40
name                                                Иван
email                                   ivan@example.com
goal                                         lose_weight
meals_per_day                                          5
diet_type                                       omnivore
restrictions                                 без арахиса
budget                                            5000.0
max_cooking_time                                    None
preferences                                         None
dtype: object

In [18]:
MEAL_PLANS = {
    3: [("Завтрак", "breakfast"), ("Обед", "lunch"), ("Ужин", "dinner")],
    4: [("Завтрак", "breakfast"), ("Обед", "lunch"), ("Полдник", "snack"), ("Ужин", "dinner")],
    5: [
        ("Завтрак", "breakfast"),
        ("2-й завтрак", "breakfast"),  # кандидаты те же, что и для завтрака
        ("Обед", "lunch"),
        ("Полдник", "snack"),
        ("Ужин", "dinner"),
    ],
}

def meal_slots(meals_per_day):
    """Return (label, meal_type) pairs for 3, 4 or 5 meals per day (other values are clamped to 3..5)."""
    count = 3 if meals_per_day is None or pd.isna(meals_per_day) else int(meals_per_day)
    count = min(max(count, 3), 5)
    return list(MEAL_PLANS[count])

slots = meal_slots(profile["meals_per_day"])
slots

[('Завтрак', 'breakfast'),
 ('2-й завтрак', 'breakfast'),
 ('Обед', 'lunch'),
 ('Полдник', 'snack'),
 ('Ужин', 'dinner')]

In [19]:
def get_eligible_meals(profile, meal_type):
    """Apply hard profile filters before vector similarity ranking."""
    conditions = [
        "m.status = 'published'",
        "m.embedding IS NOT NULL",
        "m.diet_type = ANY(%s)",
        "m.meal_type = %s",
    ]
    params = [allowed_diet_types(profile.get("diet_type")), meal_type]
    goal = normalize_goal(profile.get("goal"))
    if goal:
        conditions.append("m.goal = %s")
        params.append(goal)
    max_cooking_time = profile.get("max_cooking_time")
    if max_cooking_time is not None and not pd.isna(max_cooking_time):
        conditions.append("m.cooking_time <= %s")
        params.append(int(max_cooking_time))
    excluded = restriction_patterns(profile.get("restrictions"))
    if excluded:
        conditions.append(
            """NOT EXISTS (
                SELECT 1
                FROM unnest(COALESCE(m.allergens, ARRAY[]::text[])) AS a(value)
                WHERE lower(a.value) LIKE ANY(%s::text[])
            )"""
        )
        params.append(excluded)
        conditions.append(
            """NOT EXISTS (
                SELECT 1
                FROM meal_products AS mp
                JOIN products AS p ON p.id = mp.product_id
                WHERE mp.meal_id = m.id
                  AND (
                      lower(p.name) LIKE ANY(%s::text[])
                      OR lower(p.category) LIKE ANY(%s::text[])
                  )
            )"""
        )
        params.extend([excluded, excluded])
    query = f"""
        SELECT
            m.id,
            m.name,
            m.meal_type,
            m.calories,
            m.protein,
            m.fat,
            m.carbs,
            m.cost,
            m.cooking_time,
            m.servings,
            m.goal,
            m.diet_type,
            m.allergens,
            m.tags
        FROM meals AS m
        WHERE {" AND ".join(conditions)}
    """
    return pd.read_sql_query(query, conn, params=params)

def retrieve_candidates(query, profile, meal_type, limit=5):
    """Rank only meals that have already passed goal/diet/restriction filters."""
    eligible = get_eligible_meals(profile, meal_type)
    if eligible.empty:
        return eligible.assign(similarity=pd.Series(dtype=float))
    query_embedding = model.encode(query, normalize_embeddings=True)
    eligible_ids = [str(value) for value in eligible["id"].tolist()]
    ranked = pd.read_sql_query(
        """
        SELECT
            m.id,
            m.name,
            m.meal_type,
            m.calories,
            m.protein,
            m.fat,
            m.carbs,
            m.cost,
            m.cooking_time,
            m.servings,
            m.goal,
            m.diet_type,
            m.allergens,
            m.tags,
            1 - (m.embedding <=> %s::vector) AS similarity
        FROM meals AS m
        WHERE m.id = ANY(%s::uuid[])
          AND m.embedding IS NOT NULL
        ORDER BY m.embedding <=> %s::vector
        LIMIT %s
        """,
        conn,
        params=(query_embedding, eligible_ids, query_embedding, limit),
    )
    return ranked

In [20]:
def retrieve_candidates_by_slot(profile, extra_request="", candidates_per_type=5):
    groups = []
    uuid_by_temporary_id = {}
    next_id = 1
    goal = normalize_goal(profile.get("goal")) or "без заданной цели"
    preferences = profile.get("preferences") or ""
    first_slot_by_type = {}  # meal_type -> группа, в которой кандидаты уже найдены
    for slot_label, meal_type in meal_slots(profile.get("meals_per_day")):
        if meal_type in first_slot_by_type:
            # Повторный слот того же типа (2-й завтрак): те же кандидаты и те же ID, что у первого
            first_group = first_slot_by_type[meal_type]
            groups.append({
                "slot": slot_label,
                "meal_type": meal_type,
                "candidates": first_group["candidates"],
                "same_as": first_group["slot"],
            })
            continue
        query_parts = [slot_label, goal, preferences, extra_request]
        query = ". ".join(str(part).strip() for part in query_parts if str(part).strip())
        candidates = retrieve_candidates(
            query,
            profile,
            meal_type=meal_type,
            limit=candidates_per_type,
        )
        slot_candidates = []
        for _, row in candidates.iterrows():
            temporary_id = f"M{next_id:03d}"
            next_id += 1
            uuid_by_temporary_id[temporary_id] = str(row["id"])
            slot_candidates.append({
                "candidate_id": temporary_id,
                "name": row["name"],
                "calories": row["calories"],
                "protein": row["protein"],
                "fat": row["fat"],
                "carbs": row["carbs"],
                "price": row["cost"],
                "cooking_time": row["cooking_time"],
            })
        group = {
            "slot": slot_label,
            "meal_type": meal_type,
            "candidates": slot_candidates,
        }
        first_slot_by_type[meal_type] = group
        groups.append(group)
    return groups, uuid_by_temporary_id

candidate_groups, candidate_uuid_map = retrieve_candidates_by_slot(
    profile,
    extra_request="Сбалансированный рацион с подходящим количеством белка",
    candidates_per_type=5,
)
candidate_groups

C:\Users\valentin\AppData\Local\Temp\ipykernel_26776\3337699777.py:60: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(query, conn, params=params)
C:\Users\valentin\AppData\Local\Temp\ipykernel_26776\3337699777.py:69: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  ranked = pd.read_sql_query(
C:\Users\valentin\AppData\Local\Temp\ipykernel_26776\3337699777.py:60: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(query, conn, params=params)
C:\Users\valentin\AppData\Local\Temp\ipykern

[{'slot': 'Завтрак',
  'meal_type': 'breakfast',
  'candidates': [{'candidate_id': 'M001',
    'name': 'Картофельная фриттата',
    'calories': 214.0,
    'protein': 14.6,
    'fat': 10.8,
    'carbs': 13.8,
    'price': 45.6,
    'cooking_time': 15},
   {'candidate_id': 'M002',
    'name': 'Шакшука со шпинатом',
    'calories': 181.0,
    'protein': 14.4,
    'fat': 10.9,
    'carbs': 6.2,
    'price': 60.0,
    'cooking_time': 15},
   {'candidate_id': 'M003',
    'name': 'Банановые овсяные оладьи',
    'calories': 285.0,
    'protein': 12.6,
    'fat': 8.4,
    'carbs': 38.8,
    'price': 25.8,
    'cooking_time': 15},
   {'candidate_id': 'M004',
    'name': 'Тост с авокадо',
    'calories': 195.0,
    'protein': 7.7,
    'fat': 8.1,
    'carbs': 25.5,
    'price': 43.0,
    'cooking_time': 15},
   {'candidate_id': 'M005',
    'name': 'Каша с бананом',
    'calories': 316.0,
    'protein': 11.7,
    'fat': 7.2,
    'carbs': 51.7,
    'price': 26.7,
    'cooking_time': 15}]},
 {'slot'

In [21]:
ORDINALS = ["Первый", "Второй", "Третий", "Четвёртый", "Пятый"]

def format_budget(budget):
    if budget is None or pd.isna(budget):
        return None
    value = float(budget)
    return str(int(value)) if value.is_integer() else str(value)

def build_generation_prompt(profile, days, groups, extra_request=""):
    """Build the prompt for any number of days (1, 3, 7, ...) and 3, 4 or 5 meals per day."""
    days = int(days)
    if days < 1:
        raise ValueError("Количество дней должно быть не меньше 1")
    slots = meal_slots(profile.get("meals_per_day"))
    meals_count = len(slots)
    total_meals = days * meals_count
    total_word = "приёму" if total_meals % 10 == 1 and total_meals % 100 != 11 else "приёмам"

    budget = format_budget(profile.get("budget"))
    budget_text = (
        f"меньше или не сильно более {budget} руб./день" if budget else "без ограничения"
    )
    extra_text = str(extra_request).strip().rstrip(".") if extra_request else ""
    extra_text = extra_text or "не указаны"

    # Списки кандидатов по приёмам пищи
    candidate_lines = []
    for group in groups:
        if group.get("same_as"):
            candidate_lines.append(
                f"{group['slot']}: те же кандидаты, что и для приёма «{group['same_as']}»."
            )
            continue
        candidate_lines.append(f"{group['slot']}:")
        if not group["candidates"]:
            candidate_lines.append("  Подходящих блюд не найдено; не придумывай блюда для этого типа.")
            continue
        for item in group["candidates"]:
            candidate_lines.append(
                f"  [{item['candidate_id']}] {item['name']} — "
                f"КБЖУ: {item['calories']} ккал, Б {item['protein']} г, "
                f"Ж {item['fat']} г, У {item['carbs']} г; "
                f"цена {item['price']} руб.; готовка {item['cooking_time']} мин."
            )

    # Пример ответа: первый кандидат каждого слота (для повторного слота - тот же, что у первого)
    example_ids = []
    for index, group in enumerate(groups, start=1):
        if group["candidates"]:
            example_ids.append(group["candidates"][0]["candidate_id"])
        else:
            example_ids.append(f"M{index:03d}")
    example_meals = ", ".join(f'"{value}"' for value in example_ids)

    order_lines = [
        f"{ORDINALS[i]} candidate_id дня - {slot}." for i, (slot, _) in enumerate(slots)
    ]

    lines = [
        "Ты составляешь персональный рацион питания.",
        "Используй ТОЛЬКО candidate_id из предоставленного списка. Не придумывай и не изменяй candidate_id.",
        "Параметры плана:",
        f"Дней: {days}.",
        f"Приёмов пищи в день: {meals_count}.",
        f"Порядок приёмов пищи: {', '.join(slot for slot, _ in slots)}.",
        "Профиль пользователя:",
        f"Бюджет: {budget_text}.",
        f"Дополнительные пожелания: {extra_text}",
        "Кандидаты по приёмам пищи:",
        *candidate_lines,
        "Все предоставленные кандидаты уже прошли предварительную фильтрацию по ограничениям пользователя.",
        f"Твоя задача - распределить кандидатов по {total_meals} {total_word} пищи.",
        "Правила:",
        f"Каждый день должен содержать ровно {meals_count} candidate_id.",
        *order_lines,
        "Для каждого приёма пищи используй только candidate_id из соответствующего списка.",
        "Повторное использование candidate_id разрешено.",
    ]
    if days > 1:
        lines.append("Старайся минимизировать повторения между днями.")
    lines += [
        "Стремись соблюдать дневной бюджет.",
        "Не изменяй значения КБЖУ, цены или другие характеристики кандидатов.",
        "Если какое-либо важное ограничение невозможно выполнить одновременно с остальными, "
        "не нарушай данные кандидатов и укажи проблему в notes.",
        "Не создавай новые блюда.",
        "Не создавай новые candidate_id.",
        "Не добавляй дополнительные приёмы пищи.",
        "Приоритеты:",
        "Использовать только допустимые кандидаты.",
        "Соблюдать ограничения пользователя.",
        "Соблюдать дневной бюджет.",
        "Максимизировать разнообразие.",
        "Верни только JSON:",
        "{",
        '  "days": [',
        "    {",
        '      "day": 1,',
        f'      "meals": [{example_meals}]',
        "    }",
        "  ],",
        '  "notes": []',
        "}",
    ]
    return "\n".join(lines)

In [22]:
planned_days = 7
additional_request = "Сбалансированный рацион, простые блюда из доступных продуктов"
full_prompt = build_generation_prompt(
    profile=profile,
    days=planned_days,
    groups=candidate_groups,
    extra_request=additional_request,
)

In [23]:
print(full_prompt)

Ты составляешь персональный рацион питания.
Используй ТОЛЬКО candidate_id из предоставленного списка. Не придумывай и не изменяй candidate_id.
Параметры плана:
Дней: 7.
Приёмов пищи в день: 5.
Порядок приёмов пищи: Завтрак, 2-й завтрак, Обед, Полдник, Ужин.
Профиль пользователя:
Бюджет: меньше или не сильно более 5000 руб./день.
Дополнительные пожелания: Сбалансированный рацион, простые блюда из доступных продуктов
Кандидаты по приёмам пищи:
Завтрак:
  [M001] Картофельная фриттата — КБЖУ: 214.0 ккал, Б 14.6 г, Ж 10.8 г, У 13.8 г; цена 45.6 руб.; готовка 15 мин.
  [M002] Шакшука со шпинатом — КБЖУ: 181.0 ккал, Б 14.4 г, Ж 10.9 г, У 6.2 г; цена 60.0 руб.; готовка 15 мин.
  [M003] Банановые овсяные оладьи — КБЖУ: 285.0 ккал, Б 12.6 г, Ж 8.4 г, У 38.8 г; цена 25.8 руб.; готовка 15 мин.
  [M004] Тост с авокадо — КБЖУ: 195.0 ккал, Б 7.7 г, Ж 8.1 г, У 25.5 г; цена 43.0 руб.; готовка 15 мин.
  [M005] Каша с бананом — КБЖУ: 316.0 ккал, Б 11.7 г, Ж 7.2 г, У 51.7 г; цена 26.7 руб.; готовка 15 мин

In [24]:

import os
from pathlib import Path
from openai import OpenAI

env_file = Path("backend/.env") if Path("backend/.env").exists() else Path("../.env")

for line in env_file.read_text(encoding="utf-8").splitlines():
    line = line.strip()
    if line and not line.startswith("#") and "=" in line:
        key, value = line.split("=", 1)
        if key.strip() == "OPENROUTER_API_KEY":
            os.environ["OPENROUTER_API_KEY"] = value.strip().strip("\"'")
            break

MODELS = [
    "apodex/apodex-1.1-mini:free",
    "liquid/lfm-2.5-2.6b:free",
    "nvidia/nemotron-3-ultra-550b-a55b:free",
    "openrouter/free",
]

MODEL_INDEX = 0

if not os.environ.get("OPENROUTER_API_KEY"):
    raise ValueError("OPENROUTER_API_KEY не найден в .env")

if not 0 <= MODEL_INDEX < len(MODELS):
    raise IndexError(f"MODEL_INDEX должен быть от 0 до {len(MODELS) - 1}")

model_id = MODELS[MODEL_INDEX]

client = OpenAI(
    api_key=os.environ["OPENROUTER_API_KEY"],
    base_url="https://openrouter.ai/api/v1",
)

try:
    completion = client.chat.completions.create(
        model=model_id,
        messages=[{"role": "user", "content": full_prompt}],
        temperature=0,
    )
    model_output = completion.choices[0].message.content
    print(f"Модель: {model_id}\n")
    print(model_output)
except Exception as exc:
    print(f"Ошибка вызова {model_id}: {type(exc).__name__}: {exc}")

Модель: apodex/apodex-1.1-mini:free


{
  "days": [
    {
      "day": 1,
      "meals": ["M001", "M002", "M006", "M011", "M017"]
    },
    {
      "day": 2,
      "meals": ["M002", "M003", "M007", "M012", "M018"]
    },
    {
      "day": 3,
      "meals": ["M003", "M004", "M008", "M013", "M019"]
    },
    {
      "day": 4,
      "meals": ["M004", "M005", "M009", "M014", "M020"]
    },
    {
      "day": 5,
      "meals": ["M005", "M001", "M010", "M015", "M016"]
    },
    {
      "day": 6,
      "meals": ["M001", "M002", "M007", "M013", "M018"]
    },
    {
      "day": 7,
      "meals": ["M005", "M004", "M010", "M014", "M020"]
    }
  ],
  "notes": []
}


In [25]:
import json
import re

def parse_model_output(text):
    """Достаёт JSON из ответа модели (учитывает ```json-обёртку и лишний текст вокруг)."""
    text = re.sub(r"```(?:json)?", "", text).strip()
    start, end = text.find("{"), text.rfind("}")
    if start == -1 or end == -1:
        raise ValueError("В ответе модели не найден JSON")
    return json.loads(text[start:end + 1])

plan = parse_model_output(model_output)

# candidate_id -> блюдо, и candidate_id -> допустимые слоты
dish_by_id = {}
slots_by_id = {}
for group in candidate_groups:
    for item in group["candidates"]:
        dish_by_id[item["candidate_id"]] = item
        slots_by_id.setdefault(item["candidate_id"], set()).add(group["slot"])

slot_labels = [slot for slot, _ in meal_slots(profile["meals_per_day"])]
budget = profile.get("budget")

rows, problems = [], []
for day in plan["days"]:
    ids = day["meals"]
    if len(ids) != len(slot_labels):
        problems.append(f"День {day['day']}: {len(ids)} блюд вместо {len(slot_labels)}")
    for slot, cid in zip(slot_labels, ids):
        dish = dish_by_id.get(cid)
        if dish is None:
            problems.append(f"День {day['day']}, {slot}: неизвестный candidate_id {cid}")
            continue
        if slot not in slots_by_id[cid]:
            problems.append(f"День {day['day']}, {slot}: {cid} «{dish['name']}» относится к другому приёму пищи")
        rows.append({
            "День": day["day"], "Приём пищи": slot, "ID": cid, "Блюдо": dish["name"],
            "Ккал": dish["calories"], "Б": dish["protein"], "Ж": dish["fat"], "У": dish["carbs"],
            "Цена, ₽": dish["price"], "Готовка, мин": dish["cooking_time"],
            "meal_uuid": candidate_uuid_map.get(cid),
        })

plan_df = pd.DataFrame(rows)

# Меню по дням
print(f"Модель: {model_id}")
print(f"План для {profile['name']}: цель — {profile['goal']}, {len(plan['days'])} дн., "
      f"{len(slot_labels)} приёма(ов) пищи в день, бюджет {budget:.0f} ₽/день\n")
for day, g in plan_df.groupby("День"):
    print(f"День {day}")
    for _, r in g.iterrows():
        print(f"  {r['Приём пищи']:<8} [{r['ID']}] {r['Блюдо']} — {r['Ккал']:.0f} ккал, "
              f"Б {r['Б']} / Ж {r['Ж']} / У {r['У']}, {r['Цена, ₽']} ₽")
    print(f"  Итого: {g['Ккал'].sum():.0f} ккал, Б {g['Б'].sum():.1f} / Ж {g['Ж'].sum():.1f} / "
          f"У {g['У'].sum():.1f}, {g['Цена, ₽'].sum():.1f} ₽\n")

# Сводка по дням
summary = (plan_df.groupby("День")[["Ккал", "Б", "Ж", "У", "Цена, ₽"]].sum().round(1))
summary["В бюджете"] = summary["Цена, ₽"] <= budget
display(summary)

# Сколько раз выбрано каждое блюдо
display(plan_df.groupby(["Приём пищи", "ID", "Блюдо"]).size().rename("Раз в плане").reset_index())

# Проверки и заметки модели
print("Проверка:", "ошибок не найдено ✅" if not problems else "")
for p in problems:
    print(" ⚠️", p)
print("\nЗаметки модели:")
for note in plan.get("notes", []):
    print(" •", note)


Модель: apodex/apodex-1.1-mini:free
План для Иван: цель — lose_weight, 7 дн., 5 приёма(ов) пищи в день, бюджет 5000 ₽/день

День 1
  Завтрак  [M001] Картофельная фриттата — 214 ккал, Б 14.6 / Ж 10.8 / У 13.8, 45.6 ₽
  2-й завтрак [M002] Шакшука со шпинатом — 181 ккал, Б 14.4 / Ж 10.9 / У 6.2, 60.0 ₽
  Обед     [M006] Индейка с картофелем — 255 ккал, Б 37.2 / Ж 2.1 / У 20.8, 77.2 ₽
  Полдник  [M011] Морковь с йогуртовым дипом — 58 ккал, Б 4.6 / Ж 0.9 / У 8.2, 15.6 ₽
  Ужин     [M017] Овощной суп с нутом — 232 ккал, Б 10.3 / Ж 2.8 / У 42.7, 15.6 ₽
  Итого: 940 ккал, Б 81.1 / Ж 27.5 / У 91.7, 214.0 ₽

День 2
  Завтрак  [M002] Шакшука со шпинатом — 181 ккал, Б 14.4 / Ж 10.9 / У 6.2, 60.0 ₽
  2-й завтрак [M003] Банановые овсяные оладьи — 285 ккал, Б 12.6 / Ж 8.4 / У 38.8, 25.8 ₽
  Обед     [M007] Гречка с курицей — 560 ккал, Б 52.0 / Ж 12.0 / У 58.0, 142.0 ₽
  Полдник  [M012] Огуречные роллы с творогом — 65 ккал, Б 8.2 / Ж 2.3 / У 3.3, 16.5 ₽
  Ужин     [M018] Тунец с рисом и брокколи — 294

,Ккал,Б,Ж,У,"Цена, ₽",В бюджете
День,,,,,,
1,940.0,81.1,27.5,91.7,214.0,True
2,1385.0,117.1,35.1,145.4,347.8,True
3,1125.0,71.3,31.6,143.2,158.9,True
4,1250.0,104.4,22.2,161.7,254.0,True
5,1179.0,97.1,35.4,113.1,448.5,True
6,1404.0,118.9,37.7,143.7,364.7,True
7,1207.0,99.9,22.7,153.9,296.2,True


,Приём пищи,ID,Блюдо,Раз в плане
0,2-й завтрак,M001,Картофельная фриттата,1
1,2-й завтрак,M002,Шакшука со шпинатом,2
2,2-й завтрак,M003,Банановые овсяные оладьи,1
3,2-й завтрак,M004,Тост с авокадо,2
4,2-й завтрак,M005,Каша с бананом,1
5,Завтрак,M001,Картофельная фриттата,2
6,Завтрак,M002,Шакшука со шпинатом,1
7,Завтрак,M003,Банановые овсяные оладьи,1
8,Завтрак,M004,Тост с авокадо,1
9,Завтрак,M005,Каша с бананом,2


Проверка: ошибок не найдено ✅

Заметки модели:
